# Hate speech and LLM safety

Data preparation and analysis. Model runs are done with `llm.py` on a GPU (see README); this notebook only reads their saved predictions.

1. Data: download, clean, split, HateCheck labels, refusal requests, class balance
2. Classification results
3. Llama Guard 3 as a hate detector
4. Refusal test

In [1]:
import hashlib, html, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yaml
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

CFG = yaml.safe_load(open("config.yaml"))
SEED, LABELS = CFG["seed"], CFG["labels"]
RAW, DATA = Path("data/raw"), Path("data/processed")
PRED, PRIVATE, TABLES = Path("outputs/predictions"), Path("outputs/private"), Path("outputs/tables")
for d in (RAW, DATA, TABLES):
    d.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

## 1. Data

**Categories**
- **hate**: attacks or demeans a group of people because of a protected identity (race, religion, ethnicity, gender, sexual orientation, disability, nationality, ...)
- **offensive**: rude, vulgar or insulting, but does not target an identity group
- **neither**: everything else

**Sources** (fixed versions, see `config.yaml`)
- Davidson et al. (2017): 24,783 tweets, each labelled by 3 or more crowd workers; the label is the majority vote. Same file as the Kaggle "Hate Speech and Offensive Language" dataset.
- HateCheck (Röttger et al., 2021): 3,728 synthetic test cases in 29 functionalities (what each case tests), labelled hateful or non-hateful, for 7 target groups.
- HateCheck template placeholders: the identity terms and slurs used in the templates.

In [2]:
files = {"davidson.csv": CFG["data"]["davidson_url"],
         "hatecheck.csv": CFG["data"]["hatecheck_url"],
         "placeholders.csv": CFG["data"]["placeholders_url"]}
for name, url in files.items():
    path = RAW / name
    if not path.exists():
        path.write_bytes(requests.get(url, timeout=120).content)
    print(name, hashlib.sha256(path.read_bytes()).hexdigest()[:16])

davidson.csv fcb8bc7c68120ae4
hatecheck.csv ef1c1b95f17c1289
placeholders.csv 0ead74c40bcf633a


### Davidson: cleaning

URLs become `URL`, @mentions become `@USER`, HTML entities are decoded (some tweets are escaped twice) and whitespace is collapsed. Slurs, hashtags, emoji and spelling are kept. `agreement` is the share of annotators who chose the majority label.

In [3]:
dav = pd.read_csv(RAW / "davidson.csv", index_col=0).rename_axis("orig_id").reset_index()
dav = dav.rename(columns={"count": "n_annotators", "hate_speech": "votes_hate",
                          "offensive_language": "votes_offensive", "neither": "votes_neither",
                          "class": "label", "tweet": "text_raw"})

def clean(t):
    t = html.unescape(html.unescape(t))
    t = re.sub(r"https?://\S+|www\.\S+", "URL", t)
    t = re.sub(r"@\w+", "@USER", t)
    return re.sub(r"\s+", " ", t).strip()

dav["text"] = dav["text_raw"].map(clean)
dav["label_name"] = dav["label"].map(dict(enumerate(LABELS)))
votes = dav[["votes_hate", "votes_offensive", "votes_neither"]]
dav["agreement"] = votes.max(axis=1) / dav["n_annotators"]
dav["hate_vote_share"] = dav["votes_hate"] / dav["n_annotators"]
dav["dup_count"] = dav.groupby("text")["text"].transform("size")
print(len(dav), "tweets,", (dav.dup_count > 1).sum(), "rows share their text with another row")

24783 tweets, 310 rows share their text with another row


### Davidson: train / val / test split

Stratified 70 / 10 / 20 split by class. Identical texts are always put in the same split. The split is made on unique texts (label = most common label among the copies) and then mapped back to all rows.

The **balanced evaluation subset** is used for expensive runs: every hate post in the test split plus the same number of random offensive and neither posts.

In [4]:
uniq = dav.groupby("text")["label"].agg(lambda s: s.mode().min()).reset_index()
s = CFG["data"]["split"]
rest, test = train_test_split(uniq, test_size=s["test"], stratify=uniq["label"], random_state=SEED)
train, val = train_test_split(rest, test_size=s["val"] / (s["train"] + s["val"]),
                              stratify=rest["label"], random_state=SEED)
split_of = {**dict.fromkeys(train.text, "train"), **dict.fromkeys(val.text, "val"),
            **dict.fromkeys(test.text, "test")}
dav["split"] = dav["text"].map(split_of)

for name in ["train", "val", "test"]:
    dav[dav.split == name].to_csv(DATA / f"davidson_{name}.csv", index=False)

test_unique = dav[dav.split == "test"].drop_duplicates("text")
n_hate = (test_unique.label_name == "hate").sum()
balanced = pd.concat([g.sample(n=n_hate, random_state=SEED) for _, g in test_unique.groupby("label")])
balanced = balanced.sample(frac=1, random_state=SEED)
balanced.to_csv(DATA / "davidson_balanced.csv", index=False)

overlap = set(dav[dav.split == "train"].text) & set(dav[dav.split != "train"].text)
print("texts shared between train and val/test:", len(overlap))

texts shared between train and val/test: 0


### HateCheck: 3-class labels

- hateful → **hate**
- non-hateful profanity or abuse against non-protected targets (`profanity_nh`, `target_obj_nh`, `target_indiv_nh`, `target_group_nh`) → **offensive**
- all other non-hateful cases (reclaimed slurs, slur homonyms, counter-speech, neutral and positive statements about groups) → **neither**

The original `functionality` and `label_gold` columns are kept. HateCheck is used for evaluation only.

In [5]:
hc = pd.read_csv(RAW / "hatecheck.csv")
hc["text"] = hc["test_case"].str.strip()
hc["label_name"] = np.where(hc.label_gold == "hateful", "hate",
                   np.where(hc.functionality.isin(CFG["data"]["hatecheck_offensive"]), "offensive", "neither"))
hc.to_csv(DATA / "hatecheck.csv", index=False)

ph = pd.read_csv(RAW / "placeholders.csv")
placeholders = {r.Placeholder: [v.strip() for v in r.Values.split(",")] for r in ph.itertuples()}
hc.groupby(["label_name", "functionality"]).size().rename("n").reset_index()

,label_name,functionality,n
0,hate,derog_dehum_h,140
1,hate,derog_impl_h,140
2,hate,derog_neg_attrib_h,140
3,hate,derog_neg_emote_h,140
4,hate,negate_pos_h,140
5,hate,phrase_opinion_h,133
6,hate,phrase_question_h,140
7,hate,profanity_h,140
8,hate,ref_subs_clause_h,140
9,hate,ref_subs_sent_h,133


### Refusal test requests

Templates from `config.yaml` filled with the 7 HateCheck identity groups (or with non-identity targets and topics). Levels: hateful, violent hateful, offensive (no identity target), neutral, and identity-neutral (harmless requests that mention a group, to measure over-refusal).

In [6]:
def with_article(w):
    return ("an " if w[0].lower() in "aeiou" else "a ") + w

rows = []
for level, spec in CFG["refusal"]["levels"].items():
    if spec["targets"] == "identity":
        fills = [{"P": p, "S": with_article(s), "target": p}
                 for p, s in zip(placeholders["[IDENTITY_P]"], placeholders["[IDENTITY_S]"])]
    else:
        fills = [{"T": t, "target": t} for t in spec["targets"]]
    for source, template in spec["templates"]:
        for f in fills:
            rows.append({"level": level, "source": source, "target": f["target"],
                         "text": template.format(**f)})
req = pd.DataFrame(rows)
req.insert(0, "request_id", range(len(req)))
req.to_csv(DATA / "refusal_requests.csv", index=False)
req.groupby(["level", "source"]).size().rename("n").reset_index()

,level,source,n
0,hateful,azure_hate,7
1,hateful,llama_guard_s10,14
2,hateful,openai_hate,14
3,identity_neutral,control,42
4,neutral,neutral,42
5,offensive,untargeted,42
6,violent_hateful,azure_violence,7
7,violent_hateful,llama_guard_s1,14
8,violent_hateful,openai_threat,14


### Class balance

Imbalance ratio = size of the class divided by the size of the hate class.

In [7]:
def balance(df):
    c = df["label_name"].value_counts().reindex(LABELS, fill_value=0)
    row = {"n": int(c.sum())}
    for lab in LABELS:
        row[lab] = f"{c[lab]} ({100 * c[lab] / c.sum():.1f}%)"
    row["offensive : hate"] = round(c["offensive"] / c["hate"], 2)
    row["neither : hate"] = round(c["neither"] / c["hate"], 2)
    return row

sets = {"Davidson all": dav, "train": dav[dav.split == "train"], "val": dav[dav.split == "val"],
        "test": dav[dav.split == "test"], "balanced (from test)": balanced, "HateCheck": hc}
class_balance = pd.DataFrame({k: balance(v) for k, v in sets.items()}).T
class_balance.to_csv(TABLES / "class_balance.csv")
class_balance

,n,hate,offensive,neither,offensive : hate,neither : hate
Davidson all,24783,1430 (5.8%),19190 (77.4%),4163 (16.8%),13.42,2.91
train,17375,1005 (5.8%),13448 (77.4%),2922 (16.8%),13.38,2.91
val,2464,142 (5.8%),1910 (77.5%),412 (16.7%),13.45,2.9
test,4944,283 (5.7%),3832 (77.5%),829 (16.8%),13.54,2.93
balanced (from test),849,283 (33.3%),283 (33.3%),283 (33.3%),1.0,1.0
HateCheck,3728,2563 (68.8%),292 (7.8%),873 (23.4%),0.11,0.34


### Annotator agreement per class (Davidson)

The hate/offensive boundary is hard for people too: hate posts have much lower agreement.

In [8]:
lexicon = sorted(CFG["slurs"] + CFG["profanity"], key=len, reverse=True)
lexicon_re = re.compile(r"\b(?:" + "|".join(re.escape(w).replace(r"\ ", r"\s+") for w in lexicon)
                        + r")(?:s|es|z|ing|in|ed|er|ers|y|ty|tier|tiest|head|heads|face)?\b", re.I)

def redact(text):
    # Mask slurs and profanity for display: "bitch" -> "b****".
    return lexicon_re.sub(lambda m: m.group(0)[0] + "*" * (len(m.group(0)) - 1), text)

g = dav.groupby("label_name")
agreement = pd.DataFrame({
    "mean agreement": g.agreement.mean(),
    "unanimous %": g.agreement.apply(lambda a: 100 * (a == 1).mean()),
    "at least one hate vote %": g.votes_hate.apply(lambda v: 100 * (v > 0).mean()),
    "contains slur or profanity %": g.text.apply(lambda t: 100 * t.str.contains(lexicon_re).mean()),
}).reindex(LABELS).round(2)
agreement.to_csv(TABLES / "davidson_agreement.csv")
agreement

,mean agreement,unanimous %,at least one hate vote %,contains slur or profanity %
label_name,,,,
hate,0.73,18.39,100.00,80.70
offensive,0.92,74.76,17.25,93.86
neither,0.90,68.99,6.05,9.54


In [9]:
# Five random training examples per class, slurs masked.
ex = dav[dav.split == "train"].groupby("label_name").sample(5, random_state=SEED)
ex.assign(text=ex.text.map(redact))[["label_name", "agreement", "text"]]

,label_name,agreement,text
22384,hate,0.500000,Uncle Tom Hypocrite! URL
12485,hate,1.000000,Lmao RT @USER If u live wit ya girl and u do dishes u a f*****
13974,hate,0.666667,"RINO trash like @USER, @USER, @USER see conservatives as the Democrat Party see blacks. As their slave property. #tcot"
8989,hate,0.666667,Every s*** cop in #LosAngeles has raped or has thought about raping a white woman. That's why those half-breeds get ...
6215,hate,0.666667,@USER n**** who's b**** u f****** c****** get your gay a** out my mentions.
13723,neither,1.000000,Oreo Ice Cream Sandwich URL
15053,neither,0.666667,RT @USER: Just spent a good twenty mins trying to teach my bird the skyrim theme song. Then I realized that he was a...
19535,neither,1.000000,RT @USER: S/o all my real fans who can wait and not want me to put out trash like other rappers #signedtothestreets2...
20871,neither,1.000000,Sixten just shoved like 8 Oreos down his throat
9901,neither,1.000000,How about a Charles in Charge movie with Shia LeBouf as Charlie and Ryan Gosling as Buddy. #WhyNot.


## 2. Classification results

Each model scores the three label words after a prompt with the category definitions, zero-shot and with three examples (one per class). Predictions are in `outputs/predictions/<model>__<dataset>__<k>shot.csv`.

Metrics: per-class precision, recall and F1, macro-F1 (average F1 over the three classes, so the small hate class counts as much as the others) with a 95% bootstrap confidence interval, and the two confusion rates of interest: `hate→off` = share of true hate predicted offensive, `off→hate` = share of true offensive predicted hate.

In [10]:
def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, labels=LABELS, zero_division=0)
    m = {"n": len(y), "macro_f1": f1.mean()}
    for i, lab in enumerate(LABELS):
        m[f"{lab}_p"], m[f"{lab}_r"], m[f"{lab}_f1"] = prec[i], rec[i], f1[i]
    m["hate→off"] = (p[y == "hate"] == "offensive").mean() if (y == "hate").any() else np.nan
    m["off→hate"] = (p[y == "offensive"] == "hate").mean() if (y == "offensive").any() else np.nan
    return m

def bootstrap_ci(y, p, key="macro_f1", n=1000):
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(SEED)
    vals = [metrics(y[i], p[i])[key] for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.nanpercentile(vals, [2.5, 97.5])

balanced_ids = set(balanced.orig_id)
rows, confusions = [], {}
for path in sorted(PRED.glob("*__*shot.csv")):
    model, dataset, shots = path.stem.split("__")
    df = pd.read_csv(path)
    subsets = {dataset: df}
    if dataset == "davidson_test":
        subsets["davidson_balanced"] = df[df.id.isin(balanced_ids)]
    for name, d in subsets.items():
        lo, hi = bootstrap_ci(d.label, d.pred)
        spec = CFG["models"][model]
        rows.append({"model": model, "family": spec["family"], "size": spec["size"],
                     "dataset": name, "shots": shots, **metrics(d.label, d.pred),
                     "f1_ci": f"[{lo:.3f}, {hi:.3f}]", "label_mass": d.label_mass.mean()})
        confusions[(model, name, shots)] = confusion_matrix(d.label, d.pred, labels=LABELS)

results = pd.DataFrame(rows)
if len(results):
    results.round(3).to_csv(TABLES / "classification.csv", index=False)
    cols = ["model", "dataset", "shots", "n", "macro_f1", "f1_ci", "hate_f1", "offensive_f1",
            "neither_f1", "hate_r", "hate→off", "off→hate", "label_mass"]
    display(results[cols].round(3))

### Small vs medium model per family (macro-F1)

In [11]:
if len(results):
    sizes = results.pivot_table(index=["family", "dataset", "shots"], columns="size", values="macro_f1")
    if {"small", "medium"} <= set(sizes.columns):
        sizes["medium - small"] = sizes["medium"] - sizes["small"]
    sizes.round(3).to_csv(TABLES / "small_vs_medium.csv")
    display(sizes.round(3))

### Confusion matrices (rows = true label, columns = prediction)

In [12]:
for (model, name, shots), cm in confusions.items():
    if name == "davidson_test":
        print(f"{model}, {shots}")
        display(pd.DataFrame(cm, index=LABELS, columns=LABELS))

### HateCheck accuracy per functionality

Hate vs not hate, zero-shot. Shows which kinds of sentences each model gets wrong.

In [13]:
func = []
for path in sorted(PRED.glob("*__hatecheck__0shot.csv")):
    df = pd.read_csv(path).merge(hc[["case_id", "functionality"]], left_on="id", right_on="case_id")
    acc = ((df.pred == "hate") == (df.label == "hate")).groupby(df.functionality).mean()
    func.append(acc.rename(path.stem.split("__")[0]))
if func:
    func = pd.concat(func, axis=1).round(3)
    func.to_csv(TABLES / "hatecheck_functionality.csv")
    display(func)

## 3. Llama Guard 3 as a hate detector

Llama Guard only says safe or unsafe plus categories. A post counts as hate when it is unsafe with category S10 (Hate). `flagged %` per true class shows how often each class is flagged.

In [14]:
rows = []
for path in sorted(PRED.glob("llama-guard__*.csv")):
    if path.stem.endswith("_quick"):
        continue
    df = pd.read_csv(path)
    dataset = path.stem.split("__")[1]
    subsets = {dataset: df}
    if dataset == "davidson_test":
        subsets["davidson_balanced"] = df[df.id.isin(balanced_ids)]
    for name, d in subsets.items():
        y, p = (d.label == "hate").values, d.pred_hate.astype(bool).values
        prec, rec, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
        row = {"dataset": name, "n": len(d), "hate precision": prec, "hate recall": rec, "hate F1": f1}
        for lab in LABELS:
            row[f"flagged % {lab}"] = 100 * d.pred_hate[d.label == lab].mean()
        rows.append(row)
guard_table = pd.DataFrame(rows).round(3)
if len(guard_table):
    guard_table.to_csv(TABLES / "llama_guard.csv", index=False)
guard_table

""


## 4. Refusal test

For each model and request level: `refusal rate` = share of answers that start with a refusal (patterns in `config.yaml`), `harmful rate` = share of answers Llama Guard rates unsafe. The answers themselves stay in `outputs/private/` and are not shown.

In [15]:
judged = [pd.read_csv(p).assign(model=p.stem.split("__")[0]) for p in sorted(PRIVATE.glob("*__judged.csv"))]
if judged:
    judged = pd.concat(judged, ignore_index=True)
    agg = {"refusal rate": ("refused", "mean")}
    if "harmful" in judged:
        agg["harmful rate"] = ("harmful", "mean")
    refusal = judged.groupby(["model", "level"]).agg(**agg).round(3).unstack("level")
    refusal.to_csv(TABLES / "refusal.csv")
    display(refusal)

### Manual check of the refusal detection

A random sample of 50 answers is written to `outputs/private/handcheck.csv`. Fill the column `human_refused` with 1 or 0, then run the next cell again to see how well the automatic refusal detection agrees with the manual labels.

In [16]:
hc_path = PRIVATE / "handcheck.csv"
if len(judged) and not hc_path.exists():
    sample = judged.sample(min(50, len(judged)), random_state=SEED)[["model", "level", "text", "response", "refused"]]
    sample.assign(human_refused="").to_csv(hc_path, index=False)
if hc_path.exists():
    h = pd.read_csv(hc_path).dropna(subset=["human_refused"])
    if len(h):
        human, auto = h.human_refused.astype(int).astype(bool), h.refused.astype(bool)
        print(f"checked {len(h)}: agreement {(human == auto).mean():.2f}, "
              f"precision {(human & auto).sum() / max(auto.sum(), 1):.2f}, "
              f"recall {(human & auto).sum() / max(human.sum(), 1):.2f}")